In [1]:
#Importacion de librerias
import requests
from sqlalchemy import create_engine, text, inspect
from sqlalchemy.orm import sessionmaker
import pandas as pd
from unidecode import unidecode
import numpy as np
from tqdm import tqdm
import json
from pandas import json_normalize
import openpyxl
import concurrent.futures
from googleapiclient.discovery import build
from google.oauth2 import service_account

periodo_pago = '202609'
lider = 'SANDRA'

In [15]:
df_basepago = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\LIQUIDACION_VINCULACIONES_BOGOTA_202609.xlsx',sheet_name='Comisiones_Asesores')
df_basepago.shape

(76, 24)

In [7]:
#CREACION DE LA CONEXION CON BASE DE DATOS SQL SERVER

# Configura los detalles de la conexión
host='172.16.1.33'
port='1433'
database= 'CUN_REPOSITORIO'

try:
    # Crea la cadena de conexión
   #url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=SQL+Server'.format(host=host, port=port, db=database)
   url = 'mssql+pyodbc://@{host}:{port}/{db}?trusted_connection=yes&driver=SQL+Server'.format(host=host, port=port, db=database)

   
    # Crear conexion con base de datos
   engine = create_engine(url) 
   print("Conexion a la base de datos realizada") 
    
except Exception as e:
    print('Error:',e)

Conexion a la base de datos realizada


In [10]:

from sqlalchemy import text

with engine.begin() as connection:
    sql_query = text("""
        DELETE FROM cun_repositorio.aarboleda.basepago_vinculaciones_hist
        WHERE periodo_pago = :periodo_pago
          AND lider = :lider
    """)

    resultado = connection.execute(
        sql_query,
        {
            "periodo_pago": periodo_pago,
            "lider": lider
        }
    )

    print(f"Registros eliminados: {resultado.rowcount}")


Registros eliminados: 40


In [16]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'BASEPAGO_VINCULACIONES_HIST'
schema_name = 'AARBOLEDA'

# Send the data to sql server
df_basepago.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()

In [17]:
df_Validos = pd.read_excel(r'C:\Users\alejandro_arboledas\OneDrive - Corporación Unificada Nacional de Educación Superior - CUN\Documentos\PYTHON\VINCULACIONES\MODELO3\2026\LIQUIDACION_VINCULACIONES_BOGOTA_202609.xlsx',sheet_name='Validos')
df_Validos.shape

(3571, 18)

In [18]:
# Connection to database
host = '172.16.1.33'
port = '1433'
database = 'CUN_REPOSITORIO'
# username = 'swarm_cluster'  # Sustituye 'tu_usuario' por tu nombre de usuario real
# password = 'Repo2023*'  # Sustituye 'tu_contraseña' por tu contraseña real
#url = 'mssql+pyodbc://{username}:{password}@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(username=username, password=password, host=host, port=port, db=database)
url = 'mssql+pyodbc://@{host}:{port}/{db}?driver=ODBC Driver 17 for SQL Server'.format(host=host, port=port, db=database)
engine = create_engine(url, fast_executemany=True)

table_name = 'COMISIONES_VINCULACIONES_HIST'
schema_name = 'AARBOLEDA'

# Send the data to sql server
df_Validos.to_sql(
    name=table_name,
    schema=schema_name,
    con=engine,
    if_exists='replace',
    index=False,
    chunksize = 10000
)

# End connection
engine.dispose()